## Module 01: Spark Architecture & Cluster Topology

---

### 1. High-Level Overview
Apache Spark is a **distributed computing system** based on a **Master-Slave Architecture**. It allows you to process massive datasets by splitting the work across a cluster of computers.

### 2. The Core Components

#### A. The Driver (The "Master" Process)
* **Definition:** The central controller of a Spark Application.
* **Key Tasks:**
    * Runs the `main()` function and creates the `SparkSession`.
    * **DAG Creation:** Converts Python code into a logical "Direct Acyclic Graph" (The Plan).
    * **Task Scheduling:** Breaks the DAG into stages and tasks, then distributes them to Executors.
    * **Monitoring:** Tracks job progress and hosts the Spark UI.

#### B. The Cluster Manager (The "Resource Negotiator")
* **Definition:** An external service that manages the physical resources of the cluster.
* **Common Managers:** YARN (Hadoop), Kubernetes, or Spark Standalone.
* **Role:** Allocates CPU and RAM to the Spark application upon request from the Driver.

#### C. The Executor (The "Worker" Process)
* **Definition:** A distributed process that runs on Worker Nodes.
* **Key Tasks:**
    * **Execution:** Runs the specific tasks assigned by the Driver.
    * **Storage:** Manages "Storage Memory" for **Caching and Persistence**.
    * **Reporting:** Sends success/failure status and heartbeats back to the Driver.

---

### 3. Critical Concepts for the Architect

#### I. Directed Acyclic Graph (DAG)
The DAG is the logical sequence of operations Spark plans to perform. It is "Directed" (has a start and end) and "Acyclic" (no loops). It allows Spark to optimize the execution path before any data is actually touched.



#### II. Fault Tolerance (Lineage)
Spark tracks the history of how a dataset was built (Lineage). If an **Executor** fails and data is lost, the **Driver** uses the DAG/Lineage to re-compute that specific missing piece of data on a different Executor.
* **Benefit:** No need to restart the entire 10-hour job if one machine fails.

#### III. Caching vs. Persistence
* **Caching:** Storing a DataFrame in the **Executor's RAM**.
* **Persistence:** Storing data in RAM, Disk, or both.
* **Why?** It prevents the "Expensive Read" problem. Instead of reading from a slow hard drive (S3/HDFS) multiple times, Spark reads it once and keeps it in memory for instant access in later steps.



---

### 4. Summary Table

| Component | Responsibility | Analogous To |
| :--- | :--- | :--- |
| **Driver** | Planning & Coordinating | The Architect |
| **Cluster Manager** | Resource Allocation | The Site Manager |
| **Executor** | Processing & Storing | The Construction Worker |
| **Worker Node** | Physical Hardware | The Construction Site |

---

### 5. Self-Assessment (Review Questions)

1. **Who generates the DAG?** * *Answer:* The Driver.
2. **Where does Cached data live?** * *Answer:* In the Executor's Storage Memory.
3. **What happens if an Executor dies?** * *Answer:* The Driver detects it and re-runs the lost tasks on another Executor using Lineage.
4. **Is the Driver involved in actual data processing?** * *Answer:* No, it only coordinates. If it starts processing data, it creates a bottleneck (and likely crashes).

## Module 02: Lazy Evaluation & Execution Flow

---

### 1. Concept: Lazy Evaluation
In many programming languages, code is "Eager"—it runs the moment it is called. In PySpark, execution is **Lazy**. 

**Definition:** Spark does not execute the instructions immediately. Instead, it builds up a lineage of operations (the DAG) and waits until an **Action** is called before it actually processes any data.

### 2. The Two Types of Operations
To master Spark, you must distinguish between these two categories:

| Feature | Transformations | Actions |
| :--- | :--- | :--- |
| **Definition** | Operations that create a new DataFrame from an existing one. | Operations that trigger computation and return a result. |
| **Behavior** | **Lazy:** They only update the DAG. | **Eager:** They trigger the execution of the DAG. |
| **Output** | Returns a new DataFrame. | Returns a value (int, list) or writes to disk. |
| **Examples** | `filter()`, `select()`, `join()`, `groupBy()`, `map()` | `show()`, `count()`, `collect()`, `save()`, `first()` |



---

### 3. Why Lazy Evaluation? (The Benefits)

#### I. Query Optimization (The Catalyst Optimizer)
Since Spark knows the entire "plan" before it starts, it can optimize it. 
* **Predicate Pushdown:** If you filter data at the end of your script, Spark moves that filter to the very beginning (at the data source level) so it reads less data from the disk.
* **Column Pruning:** If you only use 2 columns out of 100, Spark will only read those 2 columns from the file.

#### II. Fault Tolerance
Because Spark has a recorded "recipe" (the Lineage) of how to create the data, if a machine fails, it simply looks at the recipe and re-runs only the missing parts.



---

### 4. Practical Example (The "PNR" Scenario)

Imagine you are working on your `pnr_etl_project`:

```python
# 1. Transformation (Lazy - No work done)
raw_df = spark.read.parquet("pnr_data.parquet")

# 2. Transformation (Lazy - No work done)
filtered_df = raw_df.filter(raw_df.status == "CONFIRMED")

# 3. Action (Eager - NOW Spark starts the cluster!)
print(filtered_df.count())

## Module 03: Narrow vs. Wide Dependencies (The Shuffle)

---

### 1. The Core Concept: Data Movement
In a distributed system, the "Cost" of an operation is determined by whether data stays on the same machine or has to travel across the network. This relationship between parent and child DataFrames is called a **Dependency**.

### 2. Narrow Dependencies (Low Cost)
A Narrow Dependency exists when each partition of the parent DataFrame is used by **at most one** partition of the child DataFrame.

* **Behavior:** No data movement across the network. All work happens within the Executor's local memory.
* **Analogy:** "Parallel Play." Each worker stays in their own lane and finishes their task without talking to others.
* **Operations:** `filter()`, `map()`, `union()`, `select()`, `drop()`.



### 3. Wide Dependencies (High Cost / The Shuffle)
A Wide Dependency exists when data from multiple parent partitions is needed to build a single child partition. This triggers a **Shuffle**.

* **The Shuffle:** This is the process of redistributing data across the cluster so that data with the same key (e.g., the same `pnr_id`) ends up on the same Executor.
* **Why it's slow:**
    1. **Disk I/O:** Data is often written to local disk before being moved.
    2. **Network I/O:** Massive amounts of data travel over the network cables.
    3. **Serialization:** Converting objects into bytes to send them over the wire.
* **Operations:** `groupBy()`, `join()`, `distinct()`, `repartition()`, `orderBy()`.



---

### 4. Comparison Summary

| Feature | Narrow Dependency | Wide Dependency (Shuffle) |
| :--- | :--- | :--- |
| **Data Movement** | None (Local to Executor) | High (Network Transfer) |
| **Performance** | Extremely Fast | Slower / Resource Intensive |
| **Network Impact** | Zero | High |
| **Failure Recovery** | Fast (Only re-run local task) | Slower (May need to re-shuffle) |

---

### 5. Best Practice: The "Filter First" Rule
To optimize your PNR project, always remember:
**Filter your data (Narrow) before you Join or Group it (Wide).**
* *Example:* If you only need "Confirmed" PNRs, filter them out first. This way, when the "Shuffle" happens, you are moving 100MB